# Human Resources Candidate Ranking
## Notebook 01 — Data, Relevance Labels, Duplicates, and Text Representation

Flat-file executable source; run from the repository root.

In [1]:
import hashlib
import math
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

RAW_DATA = "potential-talents.csv"
LABELLED_DATA = "potential-talents-labelled.csv"
QUERIES = ("aspiring human resources", "seeking human resources")
_TOKEN_RE = re.compile(r"[a-z0-9]+")

raw = pd.read_csv(RAW_DATA)
labelled = pd.read_csv(LABELLED_DATA)

source_columns = ["id", "job_title", "location", "connection", "fit"]
required_labelled = source_columns + ["final_revised_grade"]

assert list(raw.columns) == source_columns
assert list(labelled.columns) == required_labelled
assert len(raw) == 104 and len(labelled) == 104
assert raw["id"].is_unique and labelled["id"].is_unique
assert raw["id"].tolist() == labelled["id"].tolist()
pd.testing.assert_frame_equal(
    raw[source_columns].reset_index(drop=True),
    labelled[source_columns].reset_index(drop=True),
    check_dtype=False,
)
assert labelled["final_revised_grade"].notna().all()
assert set(labelled["final_revised_grade"].astype(int).unique()).issubset({0, 1, 2, 3})

quality_summary = pd.DataFrame({
    "check": [
        "raw rows", "labelled rows", "unique source IDs",
        "source fields unchanged in labelled file",
        "missing relevance grades", "grade values"
    ],
    "result": [
        len(raw), len(labelled), labelled["id"].nunique(),
        True, int(labelled["final_revised_grade"].isna().sum()),
        sorted(labelled["final_revised_grade"].astype(int).unique().tolist())
    ]
})
display(quality_summary)

def surface_text(text):
    text = "" if pd.isna(text) else str(text)
    text = text.lower().replace("&", " and ")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def normalize_text(text):
    text = surface_text(text)
    text = re.sub(r"\bhr\b", " human resources ", text)
    text = re.sub(r"\bentry level\b", " entrylevel ", text)
    return re.sub(r"\s+", " ", text).strip()

def tokenize(text):
    return _TOKEN_RE.findall(normalize_text(text))

def parse_connections(value):
    match = re.search(r"\d+", str(value).replace(",", ""))
    return float(match.group()) if match else np.nan

def stable_group_key(job_title, location, connection_num):
    payload = "|".join([
        normalize_text(job_title),
        normalize_text(location),
        "" if pd.isna(connection_num) else f"{float(connection_num):g}",
    ])
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()[:16]

df = labelled.copy()
df["relevance_grade"] = df["final_revised_grade"].astype(int)
df["connection_num"] = df["connection"].map(parse_connections)
df["title_norm"] = df["job_title"].map(normalize_text)
df["group_key"] = [
    stable_group_key(t, l, c)
    for t, l, c in zip(df["job_title"], df["location"], df["connection_num"])
]

grade_conflicts = (
    df.groupby("group_key")["relevance_grade"]
      .nunique()
      .loc[lambda s: s > 1]
)
assert grade_conflicts.empty, f"Conflicting grades inside duplicate groups: {grade_conflicts.to_dict()}"

groups = (
    df.sort_values("id")
      .groupby("group_key", sort=False)
      .agg(
          canonical_id=("id", "min"),
          member_ids=("id", lambda s: tuple(int(x) for x in s)),
          duplicate_count=("id", "size"),
          job_title=("job_title", "first"),
          location=("location", "first"),
          connection_num=("connection_num", "first"),
          title_norm=("title_norm", "first"),
          relevance_grade=("relevance_grade", "first"),
      )
      .reset_index()
      .sort_values("canonical_id")
      .reset_index(drop=True)
)

assert len(groups) == 53
assert groups["duplicate_count"].sum() == 104

group_summary = pd.DataFrame({
    "Quantity": [
        "Raw candidate rows",
        "Unique duplicate-safe profile groups",
        "Rows attributable to repeated profiles",
        "Duplicate groups with conflicting grades",
    ],
    "Value": [len(df), len(groups), len(df) - len(groups), len(grade_conflicts)],
})
display(group_summary)

data_dictionary = pd.DataFrame({
    "Field": ["id", "job_title", "location", "connection", "fit", "relevance_grade"],
    "Role in this project": [
        "Source-row identifier used for traceability",
        "Primary text used for ranking",
        "Retained for traceability and duplicate grouping",
        "Parsed for diagnostics and optional feature tests",
        "Original unused field",
        "Human 0–3 relevance target used for model development",
    ],
})
display(data_dictionary)

unique_labels = (
    groups["relevance_grade"].value_counts().sort_index()
    .rename("unique profiles").to_frame()
)
raw_labels = (
    df["relevance_grade"].value_counts().sort_index()
    .rename("raw rows after propagation").to_frame()
)
label_distribution = unique_labels.join(raw_labels, how="outer").fillna(0).astype(int)
label_distribution.index.name = "relevance grade"
display(label_distribution)

ax = label_distribution["unique profiles"].plot(
    kind="bar", figsize=(7, 4), title="Human relevance grades across unique profiles"
)
ax.set_ylabel("Unique profiles")
plt.tight_layout()
plt.show()

class BM25Corpus:
    def __init__(self, documents, k1=1.5, b=0.75):
        self.docs = [tokenize(x) for x in documents]
        self.k1 = float(k1)
        self.b = float(b)
        self.n_docs = len(self.docs)
        self.avgdl = np.mean([len(d) for d in self.docs])
        self.df = Counter()
        for d in self.docs:
            self.df.update(set(d))

    def score_documents(self, query):
        query_tokens = tokenize(query)
        out = np.zeros(self.n_docs, dtype=float)
        for i, doc in enumerate(self.docs):
            tf = Counter(doc)
            dl = len(doc)
            for term in query_tokens:
                df = self.df.get(term, 0)
                idf = math.log(1.0 + (self.n_docs - df + 0.5) / (df + 0.5))
                f = tf.get(term, 0)
                if f:
                    norm = 1 - self.b + self.b * dl / max(self.avgdl, 1e-12)
                    out[i] += idf * (f * (self.k1 + 1)) / (f + self.k1 * norm)
        return out

docs = groups["title_norm"].tolist()
bm25 = BM25Corpus(docs)

word_vectorizer = TfidfVectorizer(
    tokenizer=str.split, preprocessor=None, token_pattern=None,
    ngram_range=(1, 2), sublinear_tf=True
)
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True
)

word_docs = word_vectorizer.fit_transform(docs)
char_docs = char_vectorizer.fit_transform(docs)
word_queries = word_vectorizer.transform([normalize_text(q) for q in QUERIES])
char_queries = char_vectorizer.transform([normalize_text(q) for q in QUERIES])

representation_rows = []
for qi, query in enumerate(QUERIES):
    bm25_scores = bm25.score_documents(query)
    word_scores = cosine_similarity(word_docs, word_queries[qi]).ravel()
    char_scores = cosine_similarity(char_docs, char_queries[qi]).ravel()
    for i, row in groups.iterrows():
        representation_rows.append({
            "query": query,
            "canonical_id": int(row["canonical_id"]),
            "job_title": row["job_title"],
            "relevance_grade": int(row["relevance_grade"]),
            "bm25": float(bm25_scores[i]),
            "tfidf_word": float(word_scores[i]),
            "tfidf_char": float(char_scores[i]),
        })

representation = pd.DataFrame(representation_rows)
for query in QUERIES:
    display(
        representation[representation["query"] == query]
        .sort_values(["bm25", "canonical_id"], ascending=[False, True])
        .head(10)
        .reset_index(drop=True)
    )

checkpoint = {
    "raw_rows": int(len(raw)),
    "labelled_rows": int(len(labelled)),
    "unique_groups": int(len(groups)),
    "duplicate_grade_conflicts": int(len(grade_conflicts)),
}
print("CHECKPOINT", checkpoint)

CHECKPOINT {'raw_rows': 104, 'labelled_rows': 104, 'unique_groups': 53, 'duplicate_grade_conflicts': 0}
